In [26]:
# !pip install transformers torch

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load DialoGPT
model_name = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

print("Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.\n")

chat_history_ids = None

# ✅ Knowledge base (important for accuracy)
knowledge = {
    "what is ai": "Artificial Intelligence is a branch of computer science that enables machines to perform tasks that require human intelligence such as learning, reasoning, and decision-making.",
    "applications of ai": "AI is used in healthcare, self-driving cars, virtual assistants, recommendation systems, fraud detection, and robotics.",
    "what is machine learning": "Machine Learning is a subset of Artificial Intelligence that allows systems to learn from data and improve automatically without being explicitly programmed."
}

while True:
    user_input = input("You: ").lower()

    if user_input in ["exit", "quit"]:
        print("Chatbot: Goodbye! 👋")
        break

    # ✅ Step 1: Check knowledge base
    found = False
    for key in knowledge:
        if key in user_input:
            print("Chatbot:", knowledge[key])
            found = True
            break

    if found:
        continue

    # ✅ Step 2: Use DialoGPT
    user_input = "Answer professionally: " + user_input

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'
    )

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=0.7
    )

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", response)

    chat_history_ids = chat_history_ids[:, -1000:]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.

You: what is ai
Chatbot: Artificial Intelligence is a branch of computer science that enables machines to perform tasks that require human intelligence such as learning, reasoning, and decision-making.
You: applications of ai
Chatbot: AI is used in healthcare, self-driving cars, virtual assistants, recommendation systems, fraud detection, and robotics.
You: what is machine learning
Chatbot: Machine Learning is a subset of Artificial Intelligence that allows systems to learn from data and improve automatically without being explicitly programmed.
You: quit
Chatbot: Goodbye! 👋
